# Blackboard Systems: Dynamic Multi-Agent Coordination

A simpler, introductory version of the Blackboard pattern already lives at
`05_AI_Agent_Fundamentals/5. Agent Pattern/11_Advanced_Cognitive_Patterns/02_Blackboard.ipynb`.
That notebook contrasts a Blackboard system against a rigid sequential pipeline using a single
financial-news scenario, and it wires up its own LLM clients directly (`ChatOpenAI` / `ChatNebius`)
rather than this repo's shared factory.

This notebook is the **Phase 7 (Advanced Agentic Systems / Multi-Agent Orchestration) treatment**
of the same pattern. It uses `helpers.get_llm()` consistently, as this phase's other notebooks do,
and it builds a *different, more elaborate* scenario: a four-agent **research and report-drafting
team** — a **Researcher**, a **Fact-Checker**, an **Outline-Writer**, and a **Final-Writer** —
collaborating on a non-trivial, open-ended research question via a shared blackboard. The emphasis
here is on making the Controller's routing decisions *genuinely* dynamic: the path taken through
the agents depends on what the blackboard actually contains at each step (including loop-backs when
the Fact-Checker flags a finding), not on a hardcoded sequence.

### Definition

A **Blackboard System** is a multi-agent architecture in which several specialist agents
collaborate by reading from, and writing to, a shared, central data structure — the
*blackboard*. Instead of a fixed pipeline of hand-offs, a **Controller** inspects the current
state of the blackboard after every contribution and decides, from scratch each time, which
specialist should act next (or whether the task is complete).

### High-level Workflow

1. **Shared Memory (the Blackboard):** A running list of structured contributions — the research
   question, findings, fact-checks, outlines, and drafts — visible to every agent.
2. **Specialist Agents:** Independent agents, each with one narrow responsibility, that read the
   blackboard for context and post a new, structured contribution when activated.
3. **Controller:** A central agent that reads the whole blackboard, reasons about what is missing
   or unresolved, and emits a structured decision: which specialist to call next, or `FINISH`.
4. **Opportunistic Activation:** Only the specialist chosen by the Controller runs. Its output is
   appended to the blackboard, and control returns to the Controller.
5. **Iteration:** This repeats — including looping *back* to an earlier specialist when new
   information demands it — until the Controller judges the task complete.

### When to Use This Pattern

- The right order of operations **cannot be known in advance** — it depends on intermediate
  results (e.g., a fact-check failing and forcing a redo of research).
- Multiple specialists need to **converge on one shared artifact** rather than pass a baton once.
- You want **emergent, opportunistic problem-solving** instead of a rigid DAG of fixed hand-offs.

### Strengths & Weaknesses

**Strengths**
- Naturally handles conditional / non-linear task logic that a fixed sequential graph handles poorly.
- New specialists can be added without redesigning the whole flow — the Controller just needs to
  know they exist.
- The blackboard doubles as a transparent audit trail of *how* the answer was produced.

**Weaknesses**
- Correctness hinges entirely on the Controller's judgment — a weak or under-specified Controller
  prompt can loop forever or finish prematurely.
- Harder to reason about and test than a fixed pipeline, since the execution path is not static.
- Every controller turn is an extra LLM call, adding latency and cost versus a hardcoded router.

## Phase 0: Setup

In [ ]:
from dotenv import load_dotenv

load_dotenv()


In [ ]:
from typing import List, Literal, Optional, TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, StateGraph
from pydantic import BaseModel, Field

from helpers import get_llm

llm = get_llm()


## Phase 1: The Shared Blackboard

**What we are going to do:**
Define the `Contribution` shape every agent writes in, and the `BlackboardState` that threads
through the LangGraph graph. Contributions are structured dicts (`agent`, `type`, `content`)
rather than free-form strings, so the Controller (and we, reading the trace) can reason about
*what kind* of work has been done, not just that "something" was posted.

In [ ]:
class Contribution(TypedDict):
    """One structured entry posted to the shared blackboard."""

    agent: str
    type: Literal["finding", "fact_check", "outline", "draft"]
    content: str


class BlackboardState(TypedDict):
    research_question: str
    blackboard: List[Contribution]
    available_agents: List[str]
    next_agent: Optional[str]
    controller_reasoning: Optional[str]
    iterations: int


def format_blackboard(blackboard: List[Contribution]) -> str:
    """Render the blackboard as readable context for any agent's prompt."""
    if not blackboard:
        return "(empty -- no contributions yet)"
    lines = []
    for i, c in enumerate(blackboard, start=1):
        lines.append(f"{i}. [{c['agent']} | {c['type']}] {c['content']}")
    return "\n".join(lines)


def post(state: BlackboardState, agent: str, kind: str, content: str) -> dict:
    """Helper for a specialist node to append one contribution and bump the turn counter."""
    contribution: Contribution = {"agent": agent, "type": kind, "content": content}
    return {
        "blackboard": state["blackboard"] + [contribution],
        "iterations": state["iterations"] + 1,
    }


## Phase 2: Specialist Agents

**What we are going to do:**
Build four narrow specialists that each read the blackboard for context and post exactly one
new contribution when activated:

- **Researcher** — contributes a new finding, or revises one the Fact-Checker flagged.
- **Fact-Checker** — reviews the most recent unchecked finding and returns a structured verdict.
- **Outline-Writer** — once enough verified findings exist, organizes them into a report outline.
- **Final-Writer** — drafts the final report following the outline and the verified findings.

None of these agents decides what happens *next* — that is the Controller's job alone.

In [ ]:
def researcher_node(state: BlackboardState) -> dict:
    print("--- RESEARCHER: investigating ---")
    context = format_blackboard(state["blackboard"])
    system = SystemMessage(
        content=(
            "You are the Researcher on a small research team investigating the user's question. "
            "Read the shared blackboard below.\n\n"
            "If the most recent 'fact_check' entry has verdict 'needs_revision', address that "
            "SPECIFIC concern with a corrected or better-sourced finding -- do not repeat the "
            "flawed claim.\n"
            "Otherwise, contribute a NEW finding that has not already been covered on the "
            "blackboard.\n\n"
            "Write 3-5 sentences. Be specific and note any real uncertainty honestly -- do not "
            "invent precise statistics you are not confident about.\n\n"
            f"Blackboard so far:\n{context}"
        )
    )
    human = HumanMessage(content=f"Research question: {state['research_question']}")
    response = llm.invoke([system, human])
    return post(state, "Researcher", "finding", response.content)


class FactCheckResult(BaseModel):
    """Structured verdict on the most recent research finding."""

    target_summary: str = Field(description="Short paraphrase of the finding being checked")
    verdict: Literal["verified", "needs_revision"] = Field(
        description="'verified' if the finding is well-supported and clearly stated, "
        "'needs_revision' if it overstates certainty, is vague, or looks unsupported"
    )
    notes: str = Field(description="Why it was verified, or precisely what must be fixed")


fact_checker_llm = llm.with_structured_output(FactCheckResult)


def fact_checker_node(state: BlackboardState) -> dict:
    print("--- FACT-CHECKER: reviewing latest finding ---")
    findings = [c for c in state["blackboard"] if c["type"] == "finding"]
    latest_finding = findings[-1]
    context = format_blackboard(state["blackboard"])
    system = SystemMessage(
        content=(
            "You are the Fact-Checker on a small research team. Critically evaluate the MOST "
            "RECENT finding below for clarity, specificity, and whether its claims are "
            "reasonably supported. Be skeptical of vague hedging or overconfident claims.\n\n"
            f"Full blackboard for context:\n{context}"
        )
    )
    human = HumanMessage(
        content=f"Finding to review:\n[{latest_finding['agent']}] {latest_finding['content']}"
    )
    result = fact_checker_llm.invoke([system, human])
    content = f"Target: {result.target_summary}\nVerdict: {result.verdict}\nNotes: {result.notes}"
    return post(state, "Fact-Checker", "fact_check", content)


def outline_writer_node(state: BlackboardState) -> dict:
    print("--- OUTLINE-WRITER: structuring the report ---")
    verified_findings = _verified_findings(state["blackboard"])
    context = "\n\n".join(f"- {f['content']}" for f in verified_findings)
    system = SystemMessage(
        content=(
            "You are the Outline-Writer. Using only the VERIFIED findings below, produce a "
            "short numbered outline (3-5 sections) for a report answering the research "
            "question. Each section should name the finding(s) it will cover."
        )
    )
    human = HumanMessage(
        content=f"Research question: {state['research_question']}\n\nVerified findings:\n{context}"
    )
    response = llm.invoke([system, human])
    return post(state, "Outline-Writer", "outline", response.content)


def final_writer_node(state: BlackboardState) -> dict:
    print("--- FINAL-WRITER: drafting the report ---")
    verified_findings = _verified_findings(state["blackboard"])
    outline = [c for c in state["blackboard"] if c["type"] == "outline"][-1]
    findings_text = "\n\n".join(f"- {f['content']}" for f in verified_findings)
    system = SystemMessage(
        content=(
            "You are the Final-Writer. Write a concise final report (roughly 150-250 words) "
            "that follows the outline below and incorporates the verified findings. Use short "
            "headings that mirror the outline sections."
        )
    )
    human = HumanMessage(
        content=(
            f"Research question: {state['research_question']}\n\n"
            f"Outline:\n{outline['content']}\n\n"
            f"Verified findings:\n{findings_text}"
        )
    )
    response = llm.invoke([system, human])
    return post(state, "Final-Writer", "draft", response.content)


def _verified_findings(blackboard: List[Contribution]) -> List[Contribution]:
    """Findings whose most recent associated fact-check verdict is 'verified'.

    Approximated positionally: for each finding, look at fact-checks posted after it and take
    the latest one's verdict.
    """
    findings = [c for c in blackboard if c["type"] == "finding"]
    fact_checks = [c for c in blackboard if c["type"] == "fact_check"]
    verified = []
    for i, finding in enumerate(findings):
        later_checks = [
            fc for fc in fact_checks if blackboard.index(fc) > blackboard.index(finding)
        ]
        if later_checks and "Verdict: verified" in later_checks[-1]["content"]:
            verified.append(finding)
    return verified


## Phase 3: The Controller -- Dynamic Routing

**What we are going to do:**
Build the Controller as an LLM call with a structured output schema (`ControllerDecision`). It
receives the *entire* blackboard plus the completion criteria and must decide which specialist
acts next, or whether to `FINISH`. Crucially, nothing in this notebook hardcodes an order like
`Researcher -> Fact-Checker -> Outline-Writer -> Final-Writer` -- that ordering, and any
loop-backs, are decisions the Controller makes fresh at every turn by reading the blackboard.

In [ ]:
class ControllerDecision(BaseModel):
    """The Controller's routing decision for the next graph step."""

    next_agent: Literal[
        "Researcher", "Fact-Checker", "Outline-Writer", "Final-Writer", "FINISH"
    ] = Field(description="Which specialist should act next, or FINISH if the task is complete")
    reasoning: str = Field(description="A brief explanation of why, grounded in the blackboard")


controller_llm = llm.with_structured_output(ControllerDecision)

CONTROLLER_SYSTEM_PROMPT = """You are the Controller of a Blackboard multi-agent research system.
Your team has four specialists, available to you as: {agents}.

Roles:
- Researcher: posts a 'finding' contribution. Can also revise a finding the Fact-Checker flagged.
- Fact-Checker: reviews the most recent unchecked finding and posts a 'fact_check' contribution
  with verdict 'verified' or 'needs_revision'.
- Outline-Writer: once there are at least 2 VERIFIED findings, posts an 'outline' contribution.
- Final-Writer: once an outline exists, posts the final 'draft' contribution.

Completion criteria -- only choose FINISH when ALL of the following hold:
1. There are at least 2 'finding' contributions whose most recent matching 'fact_check' has
   verdict 'verified' (no unresolved 'needs_revision').
2. An 'outline' contribution exists, based on those verified findings.
3. A 'draft' contribution exists that follows the outline.

Routing rules of thumb (apply judgment, do not follow blindly if the blackboard suggests
otherwise):
- If the latest finding has not yet been fact-checked, call the Fact-Checker.
- If the latest fact_check says 'needs_revision', call the Researcher to fix that specific issue
  before doing anything else.
- If fewer than 2 verified findings exist, call the Researcher for another finding.
- Once 2+ verified findings exist and there is no outline yet, call the Outline-Writer.
- Once an outline exists and there is no draft yet, call the Final-Writer.
- Once a draft exists that satisfies the outline, choose FINISH.

Always ground your decision in the actual contents of the blackboard below, not in what "should"
happen next by default."""


def controller_node(state: BlackboardState) -> dict:
    context = format_blackboard(state["blackboard"])
    system = SystemMessage(
        content=CONTROLLER_SYSTEM_PROMPT.format(agents=state["available_agents"])
    )
    human = HumanMessage(
        content=(
            f"Research question: {state['research_question']}\n\n"
            f"Blackboard (turn {state['iterations']}):\n{context}\n\n"
            "Which agent should act next?"
        )
    )
    decision = controller_llm.invoke([system, human])
    print(f"--- CONTROLLER: -> {decision.next_agent} | {decision.reasoning} ---")
    return {"next_agent": decision.next_agent, "controller_reasoning": decision.reasoning}


## Phase 4: Assembling the Graph

**What we are going to do:**
Wire the Controller as the entry point and hub of the graph. Every specialist edges back to the
Controller after posting its contribution; the Controller's structured decision determines the
next hop via a conditional edge -- including `FINISH`, which routes to `END`.

In [ ]:
def route_from_controller(state: BlackboardState) -> str:
    return state["next_agent"]


graph_builder = StateGraph(BlackboardState)

graph_builder.add_node("Controller", controller_node)
graph_builder.add_node("Researcher", researcher_node)
graph_builder.add_node("Fact-Checker", fact_checker_node)
graph_builder.add_node("Outline-Writer", outline_writer_node)
graph_builder.add_node("Final-Writer", final_writer_node)

graph_builder.set_entry_point("Controller")

graph_builder.add_conditional_edges(
    "Controller",
    route_from_controller,
    {
        "Researcher": "Researcher",
        "Fact-Checker": "Fact-Checker",
        "Outline-Writer": "Outline-Writer",
        "Final-Writer": "Final-Writer",
        "FINISH": END,
    },
)

for specialist in ["Researcher", "Fact-Checker", "Outline-Writer", "Final-Writer"]:
    graph_builder.add_edge(specialist, "Controller")

blackboard_app = graph_builder.compile()


In [ ]:
# Optional: visualize the graph structure (requires network access to the Mermaid renderer)
from IPython.display import Image, display
from langchain_core.runnables.graph import MermaidDrawMethod

try:
    display(Image(blackboard_app.get_graph().draw_mermaid_png(draw_method=MermaidDrawMethod.API)))
except Exception as exc:
    print(f"Skipping diagram render ({exc}); the graph still compiles and runs fine.")


## Phase 5: Running the Blackboard on a Non-Trivial Research Question

**What we are going to do:**
Pose an open-ended research question with no single "obvious" answer, then stream the graph
turn-by-turn, printing the Controller's decision and each specialist's contribution as they are
posted. Watch for the Controller looping back to the Researcher if the Fact-Checker flags a
finding -- that loop-back is not scripted anywhere in this notebook; it emerges from the
Controller re-reading the blackboard each turn.

In [ ]:
research_question = (
    "What are the most effective strategies for keeping LLM-based agents reliable over "
    "long-horizon, multi-step tasks, and what open problems remain?"
)

initial_state: BlackboardState = {
    "research_question": research_question,
    "blackboard": [],
    "available_agents": ["Researcher", "Fact-Checker", "Outline-Writer", "Final-Writer"],
    "next_agent": None,
    "controller_reasoning": None,
    "iterations": 0,
}

final_state = None
for step in blackboard_app.stream(initial_state, {"recursion_limit": 40}):
    node_name, node_output = next(iter(step.items()))
    print(f"\n=== after '{node_name}' ===")
    if node_output.get("blackboard"):
        latest = node_output["blackboard"][-1]
        print(f"[{latest['agent']} | {latest['type']}]\n{latest['content']}\n")
    final_state = node_output


In [ ]:
# The final report is the last 'draft' contribution posted to the blackboard
final_report = [c for c in final_state["blackboard"] if c["type"] == "draft"][-1]
print(final_report["content"])


**Discussion of the Output:**
A typical run follows a path like:

1. **Controller -> Researcher.** The blackboard is empty, so the only sensible move is to get a
   first finding.
2. **Researcher -> Controller -> Fact-Checker.** The new finding hasn't been checked yet.
3. **Fact-Checker flags an issue.** If the finding overstates certainty or is too vague, the
   verdict comes back `needs_revision`.
4. **Controller -> Researcher (loop-back).** Because the routing rules say an unresolved
   `needs_revision` takes priority, the Controller sends the Researcher back to fix that specific
   finding -- rather than plowing ahead to the Outline-Writer. This loop-back is the clearest
   sign the system is not following a fixed sequence.
5. **Fact-Checker verifies the revision, then the Controller asks for a second finding**, which
   also gets checked (and may or may not need its own revision round).
6. **Once 2+ findings are verified, Controller -> Outline-Writer**, then **Controller ->
   Final-Writer**, and finally **FINISH** once the draft satisfies the outline.

The exact number of turns and whether any revision loop happens at all depends entirely on what
the LLM writes and how the Fact-Checker judges it -- rerunning this notebook can produce a
shorter or longer path. That variability is the point: the Controller is reasoning about the
*actual* blackboard contents on every turn, not replaying a script.

## Key Takeaways

- A **Blackboard system** replaces fixed agent hand-offs with a shared, structured data store
  that every specialist reads from and writes to.
- The **Controller** is the only component that decides sequencing, and it does so by re-reading
  the blackboard's current state on every turn -- not by following a hardcoded path.
- Structuring contributions (`agent`, `type`, `content`) instead of posting free-form strings
  makes it possible for both the Controller and downstream agents (like the Outline-Writer) to
  reason precisely about what has and hasn't been done.
- Genuine dynamism shows up as **loop-backs**: here, a `needs_revision` fact-check sends control
  back to the Researcher before the pipeline is allowed to move forward, something a fixed
  sequential graph cannot do without being explicitly coded for that exact case.
- This pattern trades predictability and cheap execution (a rigid graph) for flexibility on
  ill-structured tasks where the right next step genuinely depends on intermediate results --
  compare this notebook's research/report scenario with the simpler financial-news example in
  `05_AI_Agent_Fundamentals/5. Agent Pattern/11_Advanced_Cognitive_Patterns/02_Blackboard.ipynb`
  to see the same principle applied to a different, smaller problem.